# K-Means

按照**试次**维度进行K-Means聚类（每个数据点为 participant_id + trial_index）。

In [ ]:
# 避免KMeans内存泄漏
import os
os.environ["OMP_NUM_THREADS"] = "7"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from kneed import KneeLocator
from pathlib import Path
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 输入文件路径
data_dir = Path('output')
ai_call_features_by_trial_file = data_dir / 'ai_call_features_by_trial.csv'

# 输出文件路径
output_file = data_dir / 'trial_with_cluster.csv'

In [ ]:
# 加载数据
df = pd.read_csv(ai_call_features_by_trial_file)

# 查看数据概览
print("数据形状:", df.shape)

# 分离零调用试次和有调用试次
zero_callers = df[df['has_called'] == False].copy()
print(f"\n零调用试次数: {len(zero_callers)}")

df_called = df[df['has_called'] == True].copy()
# DEBUG: 不筛选
# df_called = df.copy()
print(f"\n有调用试次数: {len(df_called)}")

In [ ]:
# 查看数据源字段与缺失情况
all_columns = df.columns.tolist()
field_missing_report = pd.DataFrame({
    'field': all_columns,
    'dtype': df.dtypes.astype(str).values,
    'missing_count': df.isna().sum().values,
    'missing_rate': df.isna().mean().values,
})
field_missing_report = field_missing_report.sort_values(
    by=['missing_count', 'field'], ascending=[False, True]
).reset_index(drop=True)

print(f'数据源字段数: {len(all_columns)}')

print('\n缺失情况汇总（按缺失数降序）:')
print(field_missing_report.to_string(index=False))

In [ ]:
# 定义聚类特征（试次级）
feature_cols = [
    'trial_calls',  # AI调用总数
    'first_ai_time',    # AI首次调用时间
    # 'last_ai_time',     # AI末次调用时间
    # 'density_first_last', # 首尾调用间密度
    'pre_first_call_ideas', # AI首次调用前的想法数量
    # 'ai_interval_std',    # AI调用间隔的标准差
    'pre_think_time',   # AI调用前的平均思考时间
    # 'after_ai_time',    # AI调用后的平均思考时间
    'perspective_taking',   # 观点采择率
    'affected_by_ai', # 受AI影响率
    # 'mean_ai_distance', # 调用后3个回答与AI的平均距离的平均值
    # 'stage1_count',
    # 'stage2_count',
    # 'stage3_count',
]

# 先看这些聚类特征的缺失情况
feature_missing = df_called[feature_cols].isna().sum().sort_values(ascending=False)
print('聚类特征缺失统计:')
print(feature_missing)

# 对缺失样本采取完整案例策略，避免把缺失强行填到均值附近
df_model = df_called.dropna(subset=feature_cols).copy()
dropped_count = len(df_called) - len(df_model)
print(f'\n因聚类特征缺失被剔除的试次数: {dropped_count}')
print(f'保留用于聚类的试次数: {len(df_model)}')

In [ ]:
X = df_model[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
inertias = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

# 绘制手肘图和轮廓系数图
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('k')
axes[0].set_ylabel('簇内误差平方和')
axes[0].set_title('肘部法则')

axes[1].plot(K_range, silhouette_scores, 'ro-')
axes[1].set_xlabel('k')
axes[1].set_ylabel('轮廓系数')
axes[1].set_title('轮廓系数')
plt.tight_layout()
plt.show()

best_k_silhouette = K_range[np.argmax(silhouette_scores)]
print(f"轮廓系数建议 k={best_k_silhouette}")

# 手动或自动检测手肘点（例如用 kneed 库）
kl = KneeLocator(K_range, inertias, curve='convex', direction='decreasing')
best_k_elbow = kl.knee
print(f"手肘法建议 k={best_k_elbow}")

# 取均值作为最终的 k 值
if best_k_elbow is None:
    best_k = best_k_silhouette  # 如果手肘点无法检测到，就用轮廓系数的结果
else:
    best_k = int(round((best_k_silhouette + best_k_elbow) / 2))
best_k = 3
print(f"最终选择 k={best_k}")

In [ ]:
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_model['cluster'] = kmeans_final.fit_predict(X_scaled)

# 查看各簇样本量
print(df_model['cluster'].value_counts().sort_index())

In [ ]:
# 各簇特征均值和标准差

cluster_means = df_model.groupby('cluster')[feature_cols].mean()
cluster_stds = df_model.groupby('cluster')[feature_cols].std()

# 行为特征，列为簇，值显示为 均值±标准差
summary_table = pd.DataFrame(index=feature_cols)
for c in cluster_means.index:
    means = cluster_means.loc[c]
    stds = cluster_stds.loc[c]
    # 保留两位小数
    summary_table[f'簇 {c}'] = [f"{m:.2f} ± {s:.2f}" for m, s in zip(means, stds)]

display(summary_table)

In [ ]:
# 将标准化后的特征也计算均值（便于比较）
cluster_means_scaled = pd.DataFrame(kmeans_final.cluster_centers_, columns=feature_cols)

# 特征列名 → 中文映射
feature_name_map = {
    'trial_calls': 'AI调用次数',
    'first_ai_time': '首次调用时间',
    'pre_first_call_ideas': '首次调用前想法数',
    'pre_think_time': '平均调用前思考时间',
    'perspective_taking': '观点采择率',
    'affected_by_ai': '受AI影响率',
}

# 簇标签 → 中文映射
cluster_name_map = {
    0: '广泛试探型',
    1: '高效采纳型',
    2: '审慎批判型',
}

# 重命名特征列
cluster_means_scaled.columns = [feature_name_map.get(c, c) for c in cluster_means_scaled.columns]

# 重命名簇
cluster_means_scaled.index = [cluster_name_map.get(i, f'簇{i}') for i in cluster_means_scaled.index]

# 绘制热力图
plt.figure(figsize=(8, 6))
sns.heatmap(cluster_means_scaled.T, annot=True, cmap='RdBu_r', center=0,
            xticklabels=cluster_means_scaled.index, yticklabels=cluster_means_scaled.columns)
plt.title('各簇标准化特征均值')
plt.yticks(rotation=0)
plt.xticks(rotation=0)
plt.show()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 获取簇的标签和唯一值
clusters = df_model['cluster'].unique()
clusters = np.sort(clusters)  # 确保顺序

# 为每个簇分配一个颜色
colors = sns.color_palette("Dark2", n_colors=len(clusters))

# 创建散点图
plt.figure(figsize=(8, 6))
for i, cluster in enumerate(clusters):
    mask = df_model['cluster'] == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                c=[colors[i]], label=f'簇 {cluster}', alpha=0.7)

# 标注 participant_id-trial_index
enable_annotation = False
trial_labels = df_model.apply(
    lambda r: f"{int(r['participant_id'])}-{int(r['trial_index'])}" if pd.notna(r['trial_index']) else str(int(r['participant_id'])),
    axis=1
).values

if enable_annotation:
    for i, txt in enumerate(trial_labels):
        plt.text(X_pca[i, 0], X_pca[i, 1], str(txt), fontsize=8, alpha=0.8)

plt.xlabel('PC1', fontsize=14)
plt.ylabel('PC2', fontsize=14)
plt.title('试次级K-Means聚类（PCA投影）')
plt.legend()
plt.show()

In [ ]:
# t-SNE
from sklearn.manifold import TSNE
from scipy.spatial.distance import cdist
# 以之前标准化后的特征 X_scaled 为输入
tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, max_iter=1000, random_state=42, init='pca')
X_tsne = tsne.fit_transform(X_scaled)
# 获取簇标签（若没有 cluster 列则默认 0）
labels = df_model['cluster'].values if 'cluster' in df_model.columns else np.zeros(len(X_tsne), dtype=int)
clusters = np.unique(labels)
colors = sns.color_palette("Dark2", n_colors=len(clusters))
plt.figure(figsize=(8, 6))
for i, c in enumerate(clusters):
    mask = labels == c
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=[colors[i]], label=f'簇 {c}', alpha=0.7)

# 可选标注 participant_id-trial_index（默认关闭）
enable_annotation = False
if enable_annotation:
    trial_labels = df_model.apply(
        lambda r: f"{int(r['participant_id'])}-{int(r['trial_index'])}" if pd.notna(r['trial_index']) else str(int(r['participant_id'])),
        axis=1
    ).values

    # 仅标注离各簇中心最近的 top_n 个样本（在原始标准化空间中计算距离）
    top_n = 5
    centroids = kmeans_final.cluster_centers_  # shape (n_clusters, n_features)
    annotate_idx = []
    for c in clusters:
        c_mask = labels == c
        c_indices = np.where(c_mask)[0]
        # 计算该簇所有点到簇中心的欧氏距离
        dists = cdist(X_scaled[c_mask], centroids[c].reshape(1, -1)).ravel()
        # 取距离最近的 top_n 个
        top_local = c_indices[np.argsort(dists)[:top_n]]
        annotate_idx.extend(top_local.tolist())

    for i in annotate_idx:
        plt.text(X_tsne[i, 0], X_tsne[i, 1], str(trial_labels[i]), fontsize=8, alpha=0.9,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6, edgecolor='none'))

plt.xlabel('t-SNE1', fontsize=14)
plt.ylabel('t-SNE2', fontsize=14)
plt.title(f'K-Means聚类结果的t-SNE可视化')
plt.legend()
plt.show()

In [ ]:
# UMAP 降维并绘图（在 PCA 图下方）
try:
    from umap import UMAP
except Exception:
    import umap.umap_ as _umap
    UMAP = _umap.UMAP

# 使用之前标准化后的特征 X_scaled（请确保已运行前面的单元格）
reducer = UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_scaled)

# 获取簇标签（若没有 cluster 列则默认 0）
labels = df_model['cluster'].values if 'cluster' in df_model.columns else np.zeros(len(X_umap), dtype=int)
clusters = np.unique(labels)
colors = sns.color_palette("Dark2", n_colors=len(clusters))

plt.figure(figsize=(8, 6))
for i, c in enumerate(clusters):
    mask = labels == c
    plt.scatter(X_umap[mask, 0], X_umap[mask, 1], c=[colors[i]], label=f'簇 {c}', alpha=0.7)

# 可选标注 participant_id-trial_index（默认关闭）
enable_annotation = False
if enable_annotation:
    trial_labels = df_model.apply(
        lambda r: f"{int(r['participant_id'])}-{int(r['trial_index'])}" if pd.notna(r['trial_index']) else str(int(r['participant_id'])),
        axis=1
    ).values
    for i, txt in enumerate(trial_labels):
        plt.text(X_umap[i, 0], X_umap[i, 1], str(txt), fontsize=8, alpha=0.8)

plt.xlabel('UMAP1', fontsize=14)
plt.ylabel('UMAP2', fontsize=14)
plt.title('试次级UMAP降维（按簇着色）')
plt.legend()
plt.show()

In [ ]:
# 将聚类标签添加回 df_called，并标记零调用试次为单独一类（cluster = -1）
df_called = df_called.merge(
    df_model[['participant_id', 'trial_index', 'cluster']],
    on=['participant_id', 'trial_index'],
    how='left'
)
zero_callers['cluster'] = -1  # 零调用试次标记为 -1

# 合并全部试次
df_all = pd.concat([df_called, zero_callers], ignore_index=True)
df_all = df_all.sort_values(['participant_id', 'trial_index']).reset_index(drop=True)

# 保存带标签的数据
df_all.to_csv(output_file, index=False)

### 机器学习分类

In [ ]:
# 基于 SVM 进行分类，检测分类与聚类的正确性
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# 准备特征和标签
# 此处的 X_scaled 表示我们刚刚用于 KMeans 的标准化特征，df_model['cluster'] 就是 KMeans 的聚类结果
X_data = X_scaled
y_data = df_model['cluster'].values

# 划分训练集与测试集 (5% 作为训练，95% 用作测试)，基于标签进行分层抽样
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.95, random_state=42, stratify=y_data
)

# 构建并训练 SVM 模型
svm_model = SVC(kernel='rbf', random_state=42, C=1.0)
svm_model.fit(X_train, y_train)

# 进行预测
y_pred = svm_model.predict(X_test)

# 打印评估结果
acc = accuracy_score(y_test, y_pred)
print(f"SVM 预测准确率 (Accuracy): {acc:.4f}\n")
print("=== 分类报告 (Classification Report) ===")
print(classification_report(y_test, y_pred))

In [ ]:
import json
from pathlib import Path
Path('output').mkdir(exist_ok=True)
cl = {}
for c in sorted(df_model['cluster'].unique()):
    cl[str(c)] = {
        "n": int((df_model['cluster']==c).sum()),
        "means": {f: round(float(df_model.loc[df_model['cluster']==c, f].mean()),4) for f in feature_cols},
        "stds": {f: round(float(df_model.loc[df_model['cluster']==c, f].std()),4) for f in feature_cols}
    }
r = {"k": int(best_k), "silhouette": round(float(sil),4), "n_input": int(len(df_model)),
     "feature_cols": feature_cols, "feat_labels_cn": feat_labels_cn,
     "cluster_names": {"0":"广泛试探型","1":"高效采纳型","2":"审慎批判型"}, "clusters": cl}
Path('output/trial_kmeans.json').write_text(json.dumps(r, ensure_ascii=False, indent=2))
print("Exported trial_kmeans.json")